# Monte Carlo Methods — Final Project

## Credit portfolio tail risk: plain Monte Carlo vs importance sampling

This notebook estimates **expected loss (EL)**, **value-at-risk (VaR)**, and **expected shortfall (ES)** for three **LQD/HYG-style** portfolios (loaded from `output/issuer_portfolios_lqd_hyg.xlsx`). The baseline method is plain Monte Carlo under the **one-factor Gaussian copula model**. 

**Creative extension (“delta”):** In order to improve Monte Carlo efficiency and reduce variation, we apply **importance sampling** that tilts the systemic factor in the baseline model toward stressful conditions, and then correct the estimations with likelihood-ratio weights. Also, instead of fixing the importance-sampling tilt $\mu_{IS}$ by hand, we search a grid of $\mu_{IS}$ values, estimate which tilt yields the best variance reduction for ES99 vs plain MC, and then re-run the full replication study at that selected tilt—so the second-stage plots and CSVs correspond to the data-chosen proposal.

The sections below give the mathematical setup; the code implements simulation, diagnostics (including effective sample size ratio), sensitivity over $\mu_{IS}$, and writes figures/tables under `output/`.

## Environment and reproducibility

1. **Python:** 3.9+ recommended.
2. **Dependencies:** install from the project folder:
   ```bash
   pip install -r requirements.txt
   ```
   `openpyxl` is required because portfolios are read with `pd.read_excel`.
3. **Data / outputs:** input is `output/issuer_portfolios_lqd_hyg.xlsx`. It is a cleaned, issuer-aggregated portfolio make-up table preprocessed with the Bloomberg Terminal Excel add-ins, with company selection rules applied and necessary default probability data appended. The portfolio formation process will be explained in the next cell. The simulation result CSVs and diagnostic PNGs are written to `output/`.
4. **Runtime:** the full analysis is Monte Carlo–heavy (`N_batch` paths $\times$ many replications $\times$ three portfolios). On a laptop, expect several minutes; reduce `R`, `R_MU_SELECTION`, or `N_batch` for quicker debugging.

## Portfolio Formation Overview

The final credit-risk portfolios used in this project are formed from the holdings of two corporate bond ETFs: **LQD** and **HYG**. LQD is used to represent the investment-grade corporate bond universe, while HYG is used to represent the high-yield corporate bond universe. This design allows us to construct portfolios with different credit-risk profiles by varying the allocation between investment-grade and high-yield issuers.

The full data-cleaning and issuer-mapping workflow is implemented in a separate portfolio formation notebook. We do not include that notebook as the main submitted notebook because part of the process requires manual interaction with the **Bloomberg Terminal**, especially for linking bond-level holdings to parent company identifiers and obtaining issuer-level default probability information. Instead, the main notebook directly uses the cleaned portfolio files produced by that workflow, `issuer_portfolios_lqd_hyg.xlsx`.

The portfolio formation process proceeds as follows. First, raw LQD and HYG holdings files are downloaded from iShares. The holdings are filtered to keep only fixed-income bond positions, excluding cash, derivatives, and other non-bond rows. Because the ETF holdings are reported at the individual bond level, multiple bonds can correspond to the same corporate issuer. To convert the data into an issuer-level credit-risk portfolio, each bond is linked to its parent company using Bloomberg-based identifiers. Bond market values are then aggregated by parent issuer, producing issuer-level exposures for both the LQD and HYG universes.

After issuer aggregation, Bloomberg is used to obtain or enrich issuer-level information, including parent company ticker information and one-year default probability estimates. Issuers with unresolved Bloomberg identifiers or missing default probability data are excluded from the final selection universe. This ensures that each selected issuer has the key inputs needed for the Monte Carlo credit-risk simulation: issuer identity, sector, exposure weight, and probability of default.

We then construct three 50-issuer portfolios with different LQD/HYG allocation splits:

- **30% LQD / 70% HYG**
- **50% LQD / 50% HYG**
- **70% LQD / 30% HYG**

For each split, the number of issuers selected from each ETF is proportional to the target allocation. For example, the 30/70 portfolio selects 15 issuers from LQD and 35 issuers from HYG, while the 70/30 portfolio selects 35 issuers from LQD and 15 issuers from HYG. Issuer selection within each ETF is sector-stratified: sector quotas are assigned according to each sector’s market-value share in the ETF, and within each sector, issuers are ranked by market value. If a sector does not contain enough eligible issuers to satisfy its quota, the remaining slots are filled using the largest remaining issuers from the unselected pool.

Finally, portfolio weights are constructed in two steps. Within each ETF sleeve, selected issuers are weighted in proportion to their issuer-level market value. These within-sleeve weights are then scaled by the target ETF allocation. For example, in the 30/70 portfolio, selected LQD issuers collectively receive 30% total portfolio weight, while selected HYG issuers collectively receive 70% total portfolio weight. The resulting portfolio weights sum to one and are used as the exposure weights in the subsequent Monte Carlo credit-risk simulation.

## Mathematical model: one-factor Gaussian copula

For each issuer $i$, define a latent **standard normal** asset return

$$X_i = \sqrt{\rho}\, Z + \sqrt{1-\rho}\,\varepsilon_i, \qquad Z,\varepsilon_i \stackrel{\text{i.i.d.}}{\sim} \mathcal{N}(0,1)$$

so $\mathrm{Var}(X_i)=1$ and $\mathrm{Corr}(X_i,X_j)=\rho$. Default occurs when $X_i < c_i$ with $c_i = \Phi^{-1}(\mathrm{PD}_i)$, which calibrates the one-year default probability.

**Portfolio loss (USD):** if issuer $i$ defaults, loss is $\mathrm{EAD}_i \times \mathrm{LGD}$. Summing over issuers on each simulated path gives a loss random variable $L$; we estimate **EL**, **VaR**, and **ES** at 95% and 99% from samples of $L$ (plain MC) or weighted samples under IS.

## Importance sampling on the systemic factor

Plain Monte Carlo draws $Z \sim \mathcal{N}(0,1)$. Many paths yield **no** defaults, so tail quantities (99% VaR / ES) have **high variance**.

**Importance sampling** draws $Z \sim \mathcal{N}(\mu_{IS}, 1)$ with $\mu_{IS} < 0$ to **oversample** adverse systemic shocks. Each path carries the likelihood-ratio weight

$$w(Z) = \frac{\phi(Z;0,1)}{\phi(Z;\mu_{IS},1)} = \exp\!\left(-\mu_{IS} Z + \tfrac{1}{2}\mu_{IS}^2\right)$$

so that expectations under the true measure remain unbiased: $\mathbb{E}_P[f(L)] = \mathbb{E}_Q[w\,f(L)]$. We report **weighted** VaR/ES and the **Kish effective sample size** (`ESS`) to monitor weight degeneracy as $|\mu_{IS}|$ grows.

## Project “delta”: data-driven tilt selection

**Baseline workflow:** importance sampling with a **fixed** tilt $\mu_{IS}$ chosen by hand.

**This notebook’s modification:** for each portfolio we first **sweep** $\mu_{IS}$ on a grid (`MU_SELECTION_GRID`), estimate the **sampling standard error** of ES99 for each tilt (repeated `R_MU_SELECTION` batches), and form the **variance reduction ratio** vs plain Monte Carlo:

$$\text{VR}(\mu_{IS}) \approx \frac{\mathrm{Var}_{\text{rep}}[\widehat{\mathrm{ES99}}_{\text{plain}}]}{\mathrm{Var}_{\text{rep}}[\widehat{\mathrm{ES99}}_{\text{IS}}(\mu_{IS})]}$$

We then select the grid point with **largest** VR (excluding $\mu_{IS}=0$ when `EXCLUDE_MU_IS_ZERO_FOR_SELECTION` is `True`, because IS at zero coincides with plain MC and the ratio is dominated by Monte Carlo noise). The second stage runs the full `R`-replication comparison **only** at that selected $\mu_{IS}$.

**Rigorous analysis knobs:** increase `N_batch` for smoother loss distributions; increase `R` and `R_MU_SELECTION` to tighten standard errors on VR and ES99 SE; refine the grid step for a finer sensitivity study (at higher compute cost).

## Visualizations and exported tables

After the main simulation cell runs, you should see:

- **`mu_IS` selection** — table figure and CSV comparing each tilt on the grid: ES99 standard error, variance-reduction ratio vs plain MC, and effective sample size (ESS) ratio. This is the **parameter-sensitivity** view.
- **Replication study** (using the **selected** `mu_IS` only) — scatter and boxplot of ES99 across `R` runs, bar chart of ES99 SD (plain vs IS), and a histogram of likelihood-ratio weights from the last IS batch.
- **CSV files** — per-portfolio replication tables, summaries, combined `all_portfolios_*.csv`, and `all_portfolios_selected_mu_IS.csv` recording the chosen tilt per book.

Re-run the notebook to refresh figures if paths or parameters change.

In [2]:
"""
Credit portfolio loss simulation: plain Monte Carlo vs one-factor importance sampling.

Baseline: draw the systemic factor Z ~ N(0, 1). Delta: tilt Z ~ N(mu_IS, 1), correct
expectations with likelihood-ratio weights, then choose mu_IS from a grid by the
largest empirical variance reduction in ES99 (see main()).
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from pathlib import Path

# --- Reproducibility (single seed for NumPy legacy RNG) ------------------------
RNG_SEED = 42
np.random.seed(RNG_SEED)
plt.rcParams["figure.figsize"] = (10, 5)

# Printed DataFrames: fixed-point, 2 decimals, thousands separators (no scientific notation)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# --- Model and simulation hyperparameters ------------------------------------
LGD = 0.4  # loss given default (fraction of EAD); deterministic, homogeneous across names
rho = 0.2  # asset correlation in the one-factor Gaussian copula
N_batch = 20_000  # Monte Carlo paths per replication
R = 50  # independent replications for plain vs IS comparison (variance of estimators)
R_MU_SELECTION = 50  # replications per grid point when estimating ES99 SE for mu_IS sweep
MU_SELECTION_GRID = np.round(np.arange(0, -1.401, -0.1), 2)  # tilt grid for sensitivity / selection
ASSUMED_PORTFOLIO_VALUE = 100_000_000  # scales Excel weights to EAD in USD

# When True, mu_IS = 0 is not eligible for "best" selection (IS equals MC there; ratio is F-noise).
EXCLUDE_MU_IS_ZERO_FOR_SELECTION = True

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _out(filename: str) -> Path:
    return OUTPUT_DIR / filename


# -----------------------------------------------------------------------------
# Input workbook
# -----------------------------------------------------------------------------
# The uploaded workbook contains duplicate-looking sheets:
#   30_70, 50_50, 70_30  -> formulas show Excel errors such as #NAME?
#   Sheet1, Sheet2, Sheet3 -> same portfolios, but PD values are already computed
# Therefore this script reads Sheet1/Sheet2/Sheet3 by default.
#
# Put issuer_portfolios_lqd_hyg.xlsx either in the project root or in output/.
# -----------------------------------------------------------------------------
input_candidates = [
    PROJECT_ROOT / "issuer_portfolios_lqd_hyg.xlsx",
    OUTPUT_DIR / "issuer_portfolios_lqd_hyg.xlsx",
]

input_file = next((p for p in input_candidates if p.is_file()), None)
if input_file is None:
    raise FileNotFoundError(
        "Could not find issuer_portfolios_lqd_hyg.xlsx.\n"
        f"Checked:\n  {input_candidates[0].resolve()}\n  {input_candidates[1].resolve()}\n"
        "Please put the Excel workbook in the project folder or in the output folder."
    )

print(f"Using input workbook: {input_file.resolve()}")

PORTFOLIOS = [
    {"label": "30_70", "sheet_name": "Sheet1"},
    {"label": "50_50", "sheet_name": "Sheet2"},
    {"label": "70_30", "sheet_name": "Sheet3"},
]

EXCEL_ERROR_VALUES = [
    "#NAME?", "#VALUE!", "#DIV/0!", "#N/A", "#N/A N/A", "#REF!", "#NUM!", "#NULL!"
]


def load_portfolio(input_file, sheet_name):
    """Load issuer weights and PDs; build EAD (USD), default thresholds c_i, and per-name LGD loss in USD."""
    df = pd.read_excel(input_file, sheet_name=sheet_name, header=0)

    required_cols = ["Name", "PortfolioWeight", "1_year_default_probability"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(
            f"Sheet {sheet_name!r} is missing required columns: {missing_cols}.\n"
            f"Available columns are: {list(df.columns)}"
        )

    df = df[required_cols].copy()
    df = df.replace(EXCEL_ERROR_VALUES, np.nan)
    df = df.rename(columns={"1_year_default_probability": "PD"})

    # Convert numeric columns safely. Excel formula errors such as #NAME? become NaN.
    df["PD"] = pd.to_numeric(df["PD"], errors="coerce")
    df["PortfolioWeight"] = pd.to_numeric(df["PortfolioWeight"], errors="coerce")

    before_drop = len(df)
    df = df.dropna(subset=["Name", "PortfolioWeight", "PD"]).copy()
    dropped = before_drop - len(df)
    if dropped > 0:
        print(f"Warning: dropped {dropped} rows from {sheet_name} because Name, weight, or PD was missing/invalid.")

    # Keep PD in a valid range before applying norm.ppf.
    df = df[(df["PD"] > 0) & (df["PD"] < 1)].copy()
    if df.empty:
        raise ValueError(
            f"No valid rows remain in sheet {sheet_name!r}. "
            "Check whether the PD column contains numeric values rather than Excel errors."
        )

    df["EAD"] = df["PortfolioWeight"] * ASSUMED_PORTFOLIO_VALUE

    portfolio_value = df["EAD"].sum()
    weight = df["EAD"].values / portfolio_value
    c = norm.ppf(df["PD"].values)
    loss_if_default = df["EAD"].values.astype(float) * LGD

    return {
        "df": df,
        "names": df["Name"].values,
        "PD": df["PD"].values,
        "EAD": df["EAD"].values,
        "portfolio_value": portfolio_value,
        "weight": weight,
        "c": c,
        "loss_if_default": loss_if_default,
    }


def simulate_losses_plain(N, c, loss_if_default, rho):
    """One-factor Gaussian copula: X_i = sqrt(rho)*Z + sqrt(1-rho)*eps_i; default if X_i < c_i."""
    n = len(c)
    Z = np.random.normal(0, 1, size=N)
    eps = np.random.normal(0, 1, size=(N, n))

    X = np.sqrt(rho) * Z[:, None] + np.sqrt(1 - rho) * eps
    D = X < c

    return D @ loss_if_default


def simulate_losses_importance_sampling(N, c, loss_if_default, rho, mu_IS):
    """Same copula as plain MC, but Z ~ N(mu_IS,1). Likelihood ratio w = p0(Z)/q(Z) corrects expectations."""
    n = len(c)
    Z = np.random.normal(mu_IS, 1, size=N)
    eps = np.random.normal(0, 1, size=(N, n))

    X = np.sqrt(rho) * Z[:, None] + np.sqrt(1 - rho) * eps
    D = X < c
    L = D @ loss_if_default

    log_w = norm.logpdf(Z, loc=0, scale=1) - norm.logpdf(Z, loc=mu_IS, scale=1)
    w = np.exp(log_w)

    return L, w


def weighted_quantile(values, weights, q):
    """IS analogue of np.quantile: smallest x with weighted CDF >= q (self-normalized IS)."""
    values = np.asarray(values)
    weights = np.asarray(weights)

    order = np.argsort(values)
    values_sorted = values[order]
    weights_sorted = weights[order]

    cum_weights = np.cumsum(weights_sorted)
    cum_weights = cum_weights / cum_weights[-1]

    return values_sorted[np.searchsorted(cum_weights, q)]


def effective_sample_size(weights):
    """Kish ESS: equivalent number of equal-weight samples (diagnoses weight degeneracy under IS)."""
    weights = np.asarray(weights)
    return weights.sum() ** 2 / np.sum(weights**2)


def risk_metrics_plain(L, alphas=(0.95, 0.99)):
    """Equal-weight EL, VaR, ES from simulated loss vector L (USD)."""
    metrics = {"EL": L.mean()}

    for alpha in alphas:
        pct = int(alpha * 100)
        VaR = np.quantile(L, alpha)
        ES = L[L >= VaR].mean()

        metrics[f"VaR{pct}"] = VaR
        metrics[f"ES{pct}"] = ES

    return metrics


def risk_metrics_weighted(L, w, alphas=(0.95, 0.99)):
    """Self-normalized IS estimates of EL, VaR, ES plus effective sample size."""
    w_norm = w / w.sum()
    metrics = {
        "EL": np.sum(w_norm * L),
        "ESS": effective_sample_size(w),
    }

    for alpha in alphas:
        pct = int(alpha * 100)
        VaR = weighted_quantile(L, w, alpha)

        tail = L >= VaR
        ES = np.sum(w[tail] * L[tail]) / np.sum(w[tail])

        metrics[f"VaR{pct}"] = VaR
        metrics[f"ES{pct}"] = ES

    return metrics

def estimate_plain_es99_variability(portfolio):
    """Sample SD of ES99 across R_MU_SELECTION plain-MC runs (denominator for variance reduction)."""
    c = portfolio["c"]
    loss_if_default = portfolio["loss_if_default"]

    es99_values = []
    for _ in range(R_MU_SELECTION):
        L_plain = simulate_losses_plain(
            N=N_batch,
            c=c,
            loss_if_default=loss_if_default,
            rho=rho,
        )
        VaR99 = np.quantile(L_plain, 0.99)
        es99_values.append(L_plain[L_plain >= VaR99].mean())

    es99_values = np.asarray(es99_values)
    es99_sd = es99_values.std(ddof=1)
    es99_se = es99_sd / np.sqrt(R_MU_SELECTION)

    return es99_sd, es99_se


def compare_mu_is_grid(portfolio, label):
    """For each mu_IS: estimate ES99 SE via R_MU_SELECTION runs; variance reduction vs plain MC."""
    c = portfolio["c"]
    loss_if_default = portfolio["loss_if_default"]
    plain_es99_sd, plain_es99_se = estimate_plain_es99_variability(portfolio)

    rows = []

    for mu_IS in MU_SELECTION_GRID:
        es99_values = []
        ess_values = []

        for _ in range(R_MU_SELECTION):
            L_IS, w_IS = simulate_losses_importance_sampling(
                N=N_batch,
                c=c,
                loss_if_default=loss_if_default,
                rho=rho,
                mu_IS=mu_IS,
            )

            VaR99 = weighted_quantile(L_IS, w_IS, 0.99)
            tail = L_IS >= VaR99
            ES99 = np.sum(w_IS[tail] * L_IS[tail]) / np.sum(w_IS[tail])

            es99_values.append(ES99)
            ess_values.append(effective_sample_size(w_IS))

        es99_values = np.asarray(es99_values)
        ess_values = np.asarray(ess_values)

        es99_sd = es99_values.std(ddof=1)
        es99_se = es99_sd / np.sqrt(R_MU_SELECTION)

        rows.append({
            "mu_IS": mu_IS,
            "ES99_SE": es99_se,
            # Ratio of plain-MC to IS *squared* SDs of ES99 across replications (target ~1 at mu_IS=0).
            "Variance_Reduction_Ratio": plain_es99_sd**2 / es99_sd**2,
            "Effective_Sample_Ratio": ess_values.mean() / N_batch,
            "Average_ESS": ess_values.mean()
        })

    comparison = pd.DataFrame(rows)

    comparison.to_csv(_out(f"{label}_mu_IS_selection_comparison.csv"), index=False)
    save_mu_is_selection_table(label, comparison)

    return comparison


def save_mu_is_selection_table(label, comparison):
    table_df = comparison[
        [
            "mu_IS",
            "ES99_SE",
            "Variance_Reduction_Ratio",
            "Effective_Sample_Ratio",
        ]
    ].copy()

    table_df["mu_IS"] = table_df["mu_IS"].map(lambda x: f"{x:.1f}")
    table_df["ES99_SE"] = table_df["ES99_SE"].map(lambda x: f"${x:,.2f}")
    table_df["Variance_Reduction_Ratio"] = table_df[
        "Variance_Reduction_Ratio"
    ].map(lambda x: f"{x:.2f}x")
    table_df["Effective_Sample_Ratio"] = table_df[
        "Effective_Sample_Ratio"
    ].map(lambda x: f"{x:.1%}")

    table_df = table_df.rename(columns={
        "mu_IS": "mu_IS",
        "ES99_SE": "ES99 SE (USD)",
        "Variance_Reduction_Ratio": "Variance reduction",
        "Effective_Sample_Ratio": "ESS ratio",
    })

    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.axis("off")
    ax.set_title(
        f"{label}: mu_IS Selection Comparison",
        fontsize=13,
        pad=12,
    )

    table = ax.table(
        cellText=table_df.values,
        colLabels=table_df.columns,
        cellLoc="center",
        colLoc="center",
        loc="center",
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.35)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold")
            cell.set_facecolor("#EDEDED")

    plt.tight_layout()
    plt.savefig(_out(f"{label}_mu_IS_selection_comparison.png"), dpi=150, bbox_inches="tight")
    plt.show()


def summarize_replications(result_df, method_name, label):
    """Across-replication mean and SD for EL, VaR, ES (used in summary table)."""
    metric_cols = ["EL", "VaR95", "ES95", "VaR99", "ES99"]

    rows = []
    for metric in metric_cols:
        sd = result_df[metric].std(ddof=1)
        rows.append(
            {
                "Portfolio": label,
                "Method": method_name,
                "Metric": metric,
                "Mean": result_df[metric].mean(),
                f"SD across {R} runs": sd,
                "SE of mean": sd / np.sqrt(R),
            }
        )

    return pd.DataFrame(rows)


def run_replications(portfolio, label, mu_IS):
    """Run R independent plain-MC and IS batches (N_batch paths each); return last IS weights for diagnostics."""
    c = portfolio["c"]
    loss_if_default = portfolio["loss_if_default"]

    plain_results = []
    for r in range(1, R + 1):
        L_plain = simulate_losses_plain(
            N=N_batch,
            c=c,
            loss_if_default=loss_if_default,
            rho=rho,
        )

        plain_results.append(
            {
                "Portfolio": label,
                "Replication": r,
                **risk_metrics_plain(L_plain),
            }
        )

    is_results = []
    L_IS_last = None
    w_IS_last = None

    for r in range(1, R + 1):
        L_IS, w_IS = simulate_losses_importance_sampling(
            N=N_batch,
            c=c,
            loss_if_default=loss_if_default,
            rho=rho,
            mu_IS=mu_IS,
        )

        is_results.append(
            {
                "Portfolio": label,
                "Replication": r,
                **risk_metrics_weighted(L_IS, w_IS),
            }
        )

        L_IS_last = L_IS
        w_IS_last = w_IS

    return pd.DataFrame(plain_results), pd.DataFrame(is_results), L_IS_last, w_IS_last


def save_visualizations(label, plain_results, is_results, w_IS_last):
    """ES99 variability plots, SD comparison, and IS weight histogram (last replication)."""
    plain_ES99_sd = plain_results["ES99"].std(ddof=1)
    is_ES99_sd = is_results["ES99"].std(ddof=1)
    plain_ES99_se = plain_ES99_sd / np.sqrt(R)
    is_ES99_se = is_ES99_sd / np.sqrt(R)

    safe_label = label.lower()

    plt.figure(figsize=(8, 5))
    plt.scatter(
        plain_results["Replication"],
        plain_results["ES99"],
        marker="o",
        label="Plain MC ES99",
    )
    plt.scatter(
        is_results["Replication"],
        is_results["ES99"],
        marker="s",
        label="Importance Sampling ES99",
    )
    plt.axhline(
        plain_results["ES99"].mean(),
        linestyle="--",
        label=f"Plain MC mean = ${plain_results['ES99'].mean():,.0f}",
    )
    plt.axhline(
        is_results["ES99"].mean(),
        linestyle="--",
        label=f"IS mean = ${is_results['ES99'].mean():,.0f}",
    )
    plt.xlabel("Replication")
    plt.ylabel("ES 99% (USD)")
    plt.title(f"{label}: ES99 Estimates across {R} Independent Replications (USD)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(_out(f"{safe_label}_ES99_scatter_mean.png"), dpi=150, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(7, 5))
    plt.boxplot(
        [plain_results["ES99"], is_results["ES99"]],
        tick_labels=["Plain MC", "Importance Sampling"],
        showmeans=True,
    )
    plt.ylabel("ES 99% (USD)")
    plt.title(f"{label}: Comparison of ES99 Variability (USD)")
    plt.tight_layout()
    plt.savefig(_out(f"{safe_label}_ES99_boxplot.png"), dpi=150, bbox_inches="tight")
    plt.show()

    sd_comparison = pd.DataFrame(
        {
            "Method": ["Plain MC", "Importance Sampling"],
            "SD of ES99": [plain_ES99_sd, is_ES99_sd],
            "SE of ES99 mean": [plain_ES99_se, is_ES99_se],
        }
    )

    plt.figure(figsize=(7, 5))
    plt.bar(sd_comparison["Method"], sd_comparison["SD of ES99"])
    plt.ylabel("SD of ES99 (USD)")
    plt.title(f"{label}: Variance Reduction in ES99 Estimation (USD)")
    plt.tight_layout()
    plt.savefig(_out(f"{safe_label}_ES99_SD_comparison.png"), dpi=150, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(7, 5))
    plt.hist(w_IS_last, bins=80, edgecolor="white", alpha=0.7)
    plt.xlabel("Likelihood Ratio Weight")
    plt.ylabel("Frequency")
    plt.title(f"{label}: Distribution of Importance Sampling Weights")
    plt.tight_layout()
    plt.savefig(_out(f"{safe_label}_IS_weight_distribution.png"), dpi=150, bbox_inches="tight")
    plt.show()


def print_portfolio_overview(label, portfolio, mu_IS=None):
    PD = portfolio["PD"]
    weight = portfolio["weight"]

    print("\n" + "=" * 70)
    print(f"Portfolio: {label}")
    print("=" * 70)
    print(f"Issuers: {len(PD)}")
    print(f"Total EAD: ${portfolio['portfolio_value']:,.2f}")
    print(f"Total weight: {weight.sum():.6f}")
    print(f"PD range: [{PD.min():.2e}, {PD.max():.4f}]")
    mu_IS_str = "TBD (selected from grid sweep)" if mu_IS is None else f"{mu_IS:.2f}"
    print(f"Settings: LGD = {LGD}, rho = {rho}, mu_IS = {mu_IS_str}")
    print("Simulated loss L, EL, VaR, ES are in USD (EAD × LGD per default).")


def print_variance_reduction(label, plain_results, is_results):
    plain_ES99_sd = plain_results["ES99"].std(ddof=1)
    is_ES99_sd = is_results["ES99"].std(ddof=1)
    plain_ES99_se = plain_ES99_sd / np.sqrt(R)
    is_ES99_se = is_ES99_sd / np.sqrt(R)
    variance_reduction_ratio = (plain_ES99_sd**2) / (is_ES99_sd**2)

    print("\n" + "=" * 70)
    print(f"{label}: ES99 variance reduction")
    print("=" * 70)
    print(f"Plain MC ES99 SD across replications = ${plain_ES99_sd:,.2f}")
    print(f"IS ES99 SD across replications       = ${is_ES99_sd:,.2f}")
    print(f"Plain MC ES99 SE of mean             = ${plain_ES99_se:,.2f}")
    print(f"IS ES99 SE of mean                   = ${is_ES99_se:,.2f}")
    print(f"Variance reduction ratio             = {variance_reduction_ratio:.2f}")
    print(f"Average IS effective sample size     = {is_results['ESS'].mean():.0f}")


def main():
    """Per portfolio: (1) grid over mu_IS, (2) select best variance reduction, (3) full replication study."""
    all_plain_results = []
    all_is_results = []
    all_summary = []

    selection_summary_rows = []

    for config in PORTFOLIOS:
        label = config["label"]
        sheet_name = config["sheet_name"]

        portfolio = load_portfolio(input_file, sheet_name)
        print_portfolio_overview(label, portfolio, mu_IS=None)

        # Step 1: sensitivity — sweep mu_IS; compare ES99 sampling variance to plain MC.
        mu_comparison = compare_mu_is_grid(portfolio, label)
        print("\nmu_IS selection comparison")
        print(mu_comparison)

        if EXCLUDE_MU_IS_ZERO_FOR_SELECTION:
            candidates = mu_comparison[mu_comparison["mu_IS"] != 0.0].copy()
        else:
            candidates = mu_comparison.copy()
        best_idx = candidates["Variance_Reduction_Ratio"].idxmax()
        mu_IS = float(candidates.loc[best_idx, "mu_IS"])
        best_vr = candidates.loc[best_idx, "Variance_Reduction_Ratio"]
        best_ess_ratio = candidates.loc[best_idx, "Effective_Sample_Ratio"]
        best_es99_se = candidates.loc[best_idx, "ES99_SE"]

        print("\n" + "=" * 70)
        print(f"{label}: selected mu_IS from grid sweep")
        print("=" * 70)
        print(f"Selected mu_IS               = {mu_IS:.2f}")
        print(f"Variance reduction ratio     = {best_vr:.2f}x")
        print(f"ES99 SE at selected mu_IS    = ${best_es99_se:,.2f}")
        print(f"Effective sample size ratio  = {best_ess_ratio:.2%}")

        selection_summary_rows.append({
            "Portfolio": label,
            "Selected_mu_IS": mu_IS,
            "Variance_Reduction_Ratio": best_vr,
            "ES99_SE": best_es99_se,
            "Effective_Sample_Ratio": best_ess_ratio,
        })

        # Step 2: run full replications using the selected mu_IS, then produce
        # summary tables, variance-reduction prints, and visualizations.
        plain_results, is_results, L_IS_last, w_IS_last = run_replications(
            portfolio=portfolio,
            label=label,
            mu_IS=mu_IS,
        )

        summary_plain = summarize_replications(plain_results, "Plain MC", label)
        summary_IS = summarize_replications(is_results, "Importance Sampling", label)
        summary = pd.concat([summary_plain, summary_IS], ignore_index=True)

        print("\nPlain Monte Carlo results by replication")
        print(plain_results)
        print("\nImportance Sampling results by replication")
        print(is_results)
        print("\nSummary comparison")
        print(summary)
        print_variance_reduction(label, plain_results, is_results)

        save_visualizations(label, plain_results, is_results, w_IS_last)

        plain_results.to_csv(_out(f"{label}_plain_mc_replications.csv"), index=False)
        is_results.to_csv(_out(f"{label}_importance_sampling_replications.csv"), index=False)
        summary.to_csv(_out(f"{label}_mc_vs_importance_sampling_summary.csv"), index=False)

        all_plain_results.append(plain_results)
        all_is_results.append(is_results)
        all_summary.append(summary)

    selection_summary = pd.DataFrame(selection_summary_rows)
    selection_summary.to_csv(_out("all_portfolios_selected_mu_IS.csv"), index=False)
    print("\n" + "=" * 70)
    print("Selected mu_IS per portfolio")
    print("=" * 70)
    print(selection_summary)

    pd.concat(all_plain_results, ignore_index=True).to_csv(
        _out("all_portfolios_plain_mc_replications.csv"),
        index=False,
    )
    pd.concat(all_is_results, ignore_index=True).to_csv(
        _out("all_portfolios_importance_sampling_replications.csv"),
        index=False,
    )
    pd.concat(all_summary, ignore_index=True).to_csv(
        _out("all_portfolios_mc_vs_importance_sampling_summary.csv"),
        index=False,
    )

    print("\nSaved combined files under output/:")
    print(_out("all_portfolios_plain_mc_replications.csv"))
    print(_out("all_portfolios_importance_sampling_replications.csv"))
    print(_out("all_portfolios_mc_vs_importance_sampling_summary.csv"))


if __name__ == "__main__":
    main()


FileNotFoundError: Could not find issuer_portfolios_lqd_hyg.xlsx.
Checked:
  D:\Users\17738\Desktop\Spring 2026\Monte Carlo Methods\final project\issuer_portfolios_lqd_hyg.xlsx
  D:\Users\17738\Desktop\Spring 2026\Monte Carlo Methods\final project\output\issuer_portfolios_lqd_hyg.xlsx
Please put the Excel workbook in the project folder or in the output folder.

## Interpretation checklist (for your write-up)

1. **Baseline vs IS:** Do IS **ES99** replications cluster more tightly than plain MC (scatter / boxplot / SD bar)? That is the empirical variance reduction.
2. **`mu_IS` grid:** Does ES99 SE fall then rise as $|\mu_{IS}|$ increases? The **minimum SE** need not coincide with **maximum VR** if you use different tie-breakers; here we maximize VR (excluding zero when configured).
3. **ESS:** As $\mu_{IS}$ becomes more negative, ESS typically **falls** (weights more skewed). Comment on the **bias–variance trade-off**: aggressive tilting improves tail sampling until weight degeneracy hurts.
4. **Limitations:** homogeneous LGD, single systemic factor, Gaussian copula, static PDs — all standard simplifying assumptions for a Monte Carlo course project.

You can briefly paste key numbers from `all_portfolios_selected_mu_IS.csv` into your report and reference the saved PNGs for figures.